# End-to-End LLM Project with RAG, Tavily, and CrewAI

This notebook builds a complete question-answering workflow with:
- a local knowledge base from the `crew_data/` folder
- FAISS-based retrieval for RAG
- live web search through Tavily
- a CrewAI pipeline with Researcher, Writer, and Critic agents

The design follows the project brief closely, but it also adds a few quality-of-life improvements:
- `.env` loading instead of hardcoding secrets
- support for multiple source documents instead of one fixed PDF
- a cleaner CrewAI workflow that keeps the grounded evidence visible to the final reviewer
- comments written in a natural, human style


## What This Notebook Covers

Requirement check:
- `langchain`, `tavily-python`, and `groq`-compatible LangChain integration
- document loading from `crew_data/`
- chunking with `RecursiveCharacterTextSplitter`
- embeddings plus FAISS indexing
- retrieval-based QA
- Tavily web search for fresh information
- CrewAI agents for research, writing, and review
- an extra trend analyst agent
- a markdown report generator

Note on embeddings: the brief mentions OpenAI embeddings. This notebook uses `OpenAIEmbeddings` when `OPENAI_API_KEY` is present, and falls back to a local Hugging Face embedding model otherwise so the project still runs with only Groq and Tavily keys.


In [ ]:
%pip install -q crewai crewai-tools langchain langchain-community langchain-groq langchain-openai tavily-python faiss-cpu pypdf python-dotenv sentence-transformers matplotlib scikit-learn


In [ ]:
import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from tavily import TavilyClient

from crewai import Agent, Crew, Process, Task
from crewai_tools import tool

from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


In [ ]:
load_dotenv(override=True)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "crew_data"
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY. Add it to .env before running the notebook.")

if not TAVILY_API_KEY:
    raise ValueError("Missing TAVILY_API_KEY. Add it to .env before running the notebook.")

if not DATA_DIR.exists():
    raise FileNotFoundError("The crew_data folder is missing. Create it and add your documents first.")

print(f"Project directory: {PROJECT_DIR}")
print(f"Knowledge base folder: {DATA_DIR}")
print(f"Reports folder: {REPORTS_DIR}")
print("Embeddings provider:", "OpenAI" if OPENAI_API_KEY else "HuggingFace fallback")


## Step 1: Load and Chunk the Documents

The project brief says the notebook should read from a `crew_data/` folder, so this version loads every supported file in that folder instead of relying on a single hardcoded PDF.


In [ ]:
def load_documents(data_dir: Path):
    documents = []
    supported_patterns = ["*.pdf", "*.txt", "*.md"]
    ignored_names = {"readme.md"}

    for pattern in supported_patterns:
        for file_path in sorted(data_dir.rglob(pattern)):
            if file_path.name.lower() in ignored_names:
                continue

            if file_path.suffix.lower() == ".pdf":
                loader = PyPDFLoader(str(file_path))
            else:
                loader = TextLoader(str(file_path), encoding="utf-8")

            loaded_docs = loader.load()
            for doc in loaded_docs:
                doc.metadata["source"] = str(file_path.name)
            documents.extend(loaded_docs)

    if not documents:
        raise ValueError("No supported files were found in crew_data. Add a PDF, TXT, or MD file and rerun.")

    return documents


raw_documents = load_documents(DATA_DIR)

# These chunk settings are a good starting point for class-sized documents.
# They keep enough context to answer questions, while still staying retriever-friendly.
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} document units")
print(f"Created {len(chunks)} chunks for retrieval")


In [ ]:
def build_embeddings():
    if OPENAI_API_KEY:
        return OpenAIEmbeddings(model="text-embedding-3-small")

    # Fallback keeps the notebook usable even if only Groq and Tavily keys are available.
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


embeddings = build_embeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("FAISS index is ready")


## Step 2: Retrieval-Augmented QA

This section creates a standard RAG pipeline on top of the FAISS index. It is useful both for testing the vectorstore and for the student task that asks you to explore retrieval directly.


In [ ]:
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.1,
    groq_api_key=GROQ_API_KEY,
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
)


def ask_local_knowledge_base(question: str):
    result = qa_chain.invoke({"query": question})
    answer = result["result"]
    sources = []

    for doc in result["source_documents"]:
        source_name = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page")
        if page is None:
            sources.append(source_name)
        else:
            sources.append(f"{source_name} (page {page + 1})")

    return answer, sources


sample_question = "What are the main ideas discussed in the local documents?"
sample_answer, sample_sources = ask_local_knowledge_base(sample_question)
print("Question:", sample_question)
print("Answer:", sample_answer)
print("Sources:", sample_sources)


## Step 3: Tavily Web Search

Tavily handles fresh information that may not exist in the local knowledge base. This is the part that helps the system stay useful for current topics.


In [ ]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


def run_tavily_search(query: str, max_results: int = 5) -> str:
    results = tavily_client.search(query=query, max_results=max_results, search_depth="advanced")
    cleaned_results = []

    for idx, item in enumerate(results.get("results", []), start=1):
        cleaned_results.append(
            f"[{idx}] {item.get('title', 'Untitled')}\n"
            f"URL: {item.get('url', 'N/A')}\n"
            f"Content: {item.get('content', 'No summary returned.')}"
        )

    return "\n\n".join(cleaned_results)


print(run_tavily_search("latest advancements in LLMs")[:800])


## Step 4: CrewAI Tools and Core Agent Team

The original notebook tried to build a larger routing-and-grading chain, but one of its tasks accidentally asked the hallucination checker to validate the word `yes` or `no` instead of the real answer. This version keeps the workflow simpler and safer:
- the Researcher gathers grounded evidence from local documents and the web
- the Writer turns the evidence into a structured answer
- the Critic checks factual grounding, clarity, and completeness before returning the final version


In [ ]:
@tool("search_knowledge_base")
def search_knowledge_base(question: str) -> str:
    """Search the local FAISS index and return the most relevant passages."""
    matches = vectorstore.similarity_search(question, k=4)
    formatted_matches = []

    for idx, doc in enumerate(matches, start=1):
        source_name = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page")
        location = source_name if page is None else f"{source_name} (page {page + 1})"
        formatted_matches.append(f"[{idx}] {location}\n{doc.page_content}")

    return "\n\n".join(formatted_matches)


@tool("search_web")
def search_web(question: str) -> str:
    """Search the web with Tavily and return concise result summaries."""
    return run_tavily_search(question, max_results=5)


researcher = Agent(
    role="Researcher",
    goal="Find the most relevant facts from the local knowledge base and the live web when needed.",
    backstory=(
        "You are a careful research assistant. You prefer grounded evidence, "
        "you separate local-document findings from web findings, and you never invent sources."
    ),
    tools=[search_knowledge_base, search_web],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

writer = Agent(
    role="Content Writer",
    goal="Turn research notes into a clean, detailed answer that is easy to follow.",
    backstory=(
        "You are a strong technical writer. You organize information clearly, "
        "keep the answer readable, and avoid adding claims that were not researched."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

critic = Agent(
    role="Reviewer",
    goal="Check whether the final answer is accurate, complete, and properly grounded in the research.",
    backstory=(
        "You are the final quality gate. You remove weak claims, tighten vague wording, "
        "and make sure the answer actually addresses the question."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)


In [ ]:
def build_qa_crew(question: str) -> Crew:
    research_task = Task(
        description=(
            f"Research the following question: {question}\n\n"
            "Use the local knowledge base first whenever it is relevant. "
            "Use Tavily web search for recent facts, missing context, or current developments.\n"
            "Return well-organized research notes with clear source labels."
        ),
        expected_output="Structured research notes with grounded facts and source labels.",
        agent=researcher,
    )

    writing_task = Task(
        description=(
            f"Write a detailed answer for this question: {question}\n\n"
            "Use only the research notes you were given. "
            "Keep the structure clear and make sure the reasoning is easy to follow."
        ),
        expected_output="A well-structured answer based only on the research notes.",
        agent=writer,
        context=[research_task],
    )

    review_task = Task(
        description=(
            f"Review the draft answer for this question: {question}\n\n"
            "Check factual grounding, clarity, and completeness. "
            "Return the improved final answer, followed by a short 'Sources' section."
        ),
        expected_output="A polished final answer with a short Sources section.",
        agent=critic,
        context=[research_task, writing_task],
    )

    return Crew(
        agents=[researcher, writer, critic],
        tasks=[research_task, writing_task, review_task],
        process=Process.sequential,
        verbose=True,
    )


question = "How does retrieval-augmented generation help reduce hallucinations in LLM applications?"
qa_crew = build_qa_crew(question)
crew_answer = qa_crew.kickoff()
print(crew_answer)


## Student Task 1: Explore the Vectorstore


In [ ]:
query = "What documents mention AI agents or multi-agent workflows?"
results = vectorstore.similarity_search(query, k=3)

for idx, doc in enumerate(results, start=1):
    source_name = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page")
    label = source_name if page is None else f"{source_name} (page {page + 1})"
    print(f"\nResult {idx}: {label}")
    print(doc.page_content[:500])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# A small sample is enough here. The goal is to see how chunk embeddings spread out.
sample_chunks = chunks[: min(30, len(chunks))]
sample_texts = [doc.page_content for doc in sample_chunks]
sample_vectors = np.array(embeddings.embed_documents(sample_texts))

pca = PCA(n_components=2)
coordinates = pca.fit_transform(sample_vectors)

plt.figure(figsize=(8, 5))
plt.scatter(coordinates[:, 0], coordinates[:, 1], c=range(len(coordinates)), cmap="viridis", s=60)
plt.title("PCA View of Chunk Embeddings")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(label="Chunk index")
plt.tight_layout()
plt.show()


## Student Task 2: Add a New Agent

This agent focuses only on recent developments and uses Tavily as its main tool.


In [ ]:
trend_analyst = Agent(
    role="Trend Analyst",
    goal="Summarize the latest news and trends on a given AI topic.",
    backstory=(
        "You track current developments, keep the summary focused, "
        "and highlight the most relevant changes rather than dumping raw search output."
    ),
    tools=[search_web],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

trend_task = Task(
    description=(
        "Search the web and summarize the latest news on: AI agents in healthcare\n"
        "Return 5 short bullet points with the main takeaway from each."
    ),
    expected_output="Five concise bullet points covering the latest trend updates.",
    agent=trend_analyst,
)

trend_crew = Crew(agents=[trend_analyst], tasks=[trend_task], process=Process.sequential, verbose=True)
trend_summary = trend_crew.kickoff()
print(trend_summary)


## Student Task 3: Report Generator Agent

This part creates a markdown report and saves it in the `reports/` folder.


In [ ]:
report_generator = Agent(
    role="Report Generator",
    goal="Create a structured markdown report grounded in both local documents and live web findings.",
    backstory=(
        "You write practical research reports. You keep the structure sharp, "
        "call out limitations honestly, and make the report useful to a reader who wants the big picture quickly."
    ),
    tools=[search_knowledge_base, search_web],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

report_task = Task(
    description=(
        "Write a markdown report on: AI in clinical summarization\n\n"
        "Include these sections exactly:\n"
        "1. Executive Summary\n"
        "2. Key Findings\n"
        "3. Challenges and Limitations\n"
        "4. Future Outlook\n"
        "5. Conclusion\n"
        "6. Sources\n\n"
        "Use the knowledge base and web search where appropriate."
    ),
    expected_output="A complete markdown report with the requested sections.",
    agent=report_generator,
)

report_crew = Crew(agents=[report_generator], tasks=[report_task], process=Process.sequential, verbose=True)
report_output = report_crew.kickoff()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = REPORTS_DIR / f"report_{timestamp}.md"
report_path.write_text(str(report_output), encoding="utf-8")

print(f"Saved report to {report_path}")
print(str(report_output)[:1200])


## Optional: Interactive Question Loop

Run this only when you want to keep asking questions in one session.


In [ ]:
def ask_with_agents(question: str):
    crew = build_qa_crew(question)
    return crew.kickoff()


print("Ask a question about your local documents or a current topic. Leave it blank to stop.\n")

while True:
    user_question = input("Question: ").strip()
    if not user_question:
        print("Session ended.")
        break

    print("\nAnswer:\n")
    print(ask_with_agents(user_question))
    print("\n" + "-" * 80 + "\n")


## Final Notes

What changed compared with the earlier version:
- local data now comes from `crew_data/` as required
- the vectorstore is FAISS-based and reusable across tasks
- `.env` secrets are loaded safely instead of being described as hardcoded values
- the CrewAI flow now uses the exact Researcher, Writer, and Critic roles requested in the brief
- the trend analyst and report generator tasks are included
- comments and markdown were rewritten to sound natural and clear

If you want to extend the project further, the next sensible upgrades are:
- swap FAISS for Chroma and compare retrieval quality
- add a Streamlit or Flask interface
- export reports to PDF
- test different chunk sizes, overlap values, and embedding models
